# Task 4 — Extensions (40%)

This task is intentionally open-ended. There is no single correct answer and no fixed
set of steps to follow. You are expected to **choose a direction, justify it, implement
it, and evaluate it critically**.

40% of the total project grade comes from this task. Marks are awarded for:

- **Depth of understanding** — can you explain what you did and why it works?
- **Correctness and verification** — did you check that your implementation is right?
- **Experimental rigour** — are your results backed by plots, numbers, or comparisons?
- **Independence** — did you go beyond what was handed to you in Tasks 1–3?

> You are **not** required to attempt all directions below. One direction explored
> deeply is worth far more than five directions touched superficially.

I have also provided an expected difficulty for each of the directions, so that students
may allocate time appropriately. While harder directions are more impressive, it is 
much better to explore a direction *well* rather than do a poor/incomplete job on
a more difficult task. If students attempt an easy task and have extra time, they
are free to attempt another easy task and submit that too.

## What is graded

The main deliverable for Task 4 is your **report** (max 5 pages, A4, 12pt, 2.5 cm margins).
Code and notebooks are supporting evidence and *should* be submitted, but we are not obliged
to grade them.

For each direction you explore, your report should cover some of the following:

1. **Method** — what did you implement or experiment with? Explain the key ideas in your own words.
2. **Verification** — how do you know it is correct? (unit tests, sanity checks, known results, etc.)
3. **Results** — quantitative evidence: convergence curves, win rates, runtimes, SPR values.
4. **Reflection** — what worked, what did not, and why?

It is perfectly fine if something you tried did not work. A thoughtful post-mortem
is more valuable than a working implementation with no explanation.


## Direction A — Solving the full multi-round game (medium)

Tasks 1–3 solve a *single round* of tiny_hana. The full game is played over multiple
rounds: each round's outcome shifts the favor state, which is the starting condition
for the next round. This creates a **stochastic game** (Markov game) where a player's
optimal single-round strategy depends on the current favor state.

The single-round Nash strategy computed in Task 3 is optimal only when rounds are played independently. 
When favor state carries over, players may seek not necessarily to win the current round,
but put themselves in a favorable position for the next. We have not accounted for this
in the single-round variant.

**Possible approach — value iteration over Markov states:**
- The Markov state is `(starting_player, favor_dict)`.
- For each Markov state, solve a single-round subgame whose leaf payoffs include the
  estimated future value of the resulting next Markov state.
- Iterate until the value estimates converge.

**Questions to explore:**
- How many distinct Markov states exist? Does the value function converge quickly?
- How does the multi-round Nash strategy differ from repeatedly playing the single-round Nash?
- Can you measure exploitability in the multi-round game?
- Can you speed up solvers for these games, possibly by exploiting additional structure?


## Direction B — Scaling to medium_hana (hard)

medium_hana adds a third action type (`DISCARD`) and five geishas instead of three,
making the game tree significantly larger.

The treeplex and CFR implementation from Tasks 2–3 should generalise
to medium_hana with minimal changes (swap `tiny_hana` for `medium_hana` in `env_config.py`).
The key question is whether it *scales* in practice.

**Questions to explore:**
- How much larger is the medium_hana treeplex? How does CFR convergence compare?
- Does a Nash agent for medium_hana play noticeably differently from the heuristic agents
  in `medium_hana/example_agents.py`?
- Are there structural symmetries in medium_hana you can exploit to reduce the problem size?

**Extra hard:**
- Can we solve the multi-round variant of the game for medium_hana?


## Direction C — Algorithmic improvements to CFR (easy)

Vanilla CFR converges at O(1/√T). Several variants achieve faster practical convergence.

| Variant | Key idea |
|---------|----------|
| **CFR+** | Replace cumulative regrets with `max(R, 0)` |
| **Linear CFR (LCFR)** | Weight iteration t with factor t, so recent iterations count more |
| **MCCFR** | Sample only a subset of the tree per iteration — scales to very large games |
| **Predictive / optimistic CFR** | Use the previous gradient as a prediction to reduce regret |

**Questions to explore:**
- Implement one or more variants as subclasses of `CFRSolver`.
- Plot SPR vs. iteration count for vanilla CFR and your variant on the same axes.
- Does the improvement hold across different games (tiny vs medium)?


## Direction D — Alternative Nash solvers (medium)

Try another solver and compare their performance with what you submitted in Task 3.

For instance, if you used CFR in Task 3, you may choose to try the following for Task 4.

**Sequence-form LP:**  
The Nash equilibrium of a two-player zero-sum extensive-form game can be computed
exactly by solving a linear program over the sequence-form strategy space. This is
exact but does not scale as well as CFR for large games.

**Double Oracle / PSRO:**  
Maintain a small restricted strategy set for each player and iteratively add best
responses. The restricted game is solved at each step (e.g., via a small LP or regret minimization+self-play).

**Questions to explore:**
- Implement one alternative solver and compare its convergence and runtime to CFR.
- For the linear programming: verify that the solution matches the CFR solution.
- For the PSRO, what is your termination criterion?
- Where does each method break down as the game gets larger?


## Direction E — Approximate and learning-based methods (hard)

For very large games, exact methods are infeasible. Approximate approaches trade
solution quality for scalability.

| Approach | Idea |
|----------|------|
| **Deep CFR** | Approximate the regret/strategy with a neural network instead of a table |
| **MARL** (multi-agent RL) | Train agents via self-play with policy gradient or Q-learning |
| **Subgame solving** | Solve subgames more carefully when reached during play |
| **Abstraction** | Merge similar information sets to reduce the game size |

**Questions to explore:**
- Does a learned agent approach the exploitability of the exact Nash solution?
- How much training data / compute is needed relative to exact CFR?
- What information-set features are most useful for the function approximator? 
- For MARL methods, are you using a centralized or decentralized approach?
- Can you evaluate your agent by estimating exploitability when these methods are used? Note that just saying 'we ran RL and self-play and things converged' is not a good measure of exploitability.

> Note: these methods are significantly more involved. Make sure you have Tasks 1–3
> solid before attempting this direction.


## Direction F — Exploiting game structure (easy)

tiny_hana has symmetries and structure that can be exploited to speed up solving.

**Examples:**
- **Player symmetry:** the game is symmetric between players. Can you solve for one
  player and derive the other's strategy for free?
- **Suit permutation symmetry:** permuting suits that have the same rank yields an
  equivalent game. Can you deduplicate information sets or treeplex nodes accordingly?
- **Caching:** certain subgames repeat across different Markov states. Can you
  identify and reuse their solutions?

**Questions to explore:**
- Quantify the reduction in treeplex size from exploiting a symmetry.
- Does the resulting strategy still achieve the same SPR as the original?
- How does the speedup scale with game size?


## Tips

**Starting point.** Your Task 3 implementation is the natural foundation for most
directions above. Keep it working and build on top of it.

**Measuring progress.** Always have a quantitative metric. SPR / exploitability is
the standard for Nash-quality; win rate against a fixed opponent is useful for
heuristic comparisons.

**What not to do.** Tuning hyperparameters without understanding why they help is
not a meaningful experiment. Neither is running the same algorithm longer without
a hypothesis about what you expect to see.

**Discussing with the lecturer.** If you want to explore a direction not listed here,
check in first. Unusual directions are welcome as long as they connect meaningfully
to the course material.
